## 1. Import libraries

In [1]:
import numpy as np
import pandas as pd

In this step, the dataset is loaded from a CSV file.  
Each row represents one user who participated in the A/B test.

## 2. Load data

In [2]:
data = pd.read_csv("C:/Users/glazk/Downloads/ab_test_results.csv",
    low_memory=False)

In [3]:
data.head() # Verify loading

,user_id,variant,converted,session_duration,pages_visited,age
0,1,control,0,3.494324,7,57
1,2,treatment,0,4.452306,4,24
2,3,control,0,4.384545,1,49
3,4,control,0,6.338822,6,36
4,5,control,0,1.576521,3,23


## 3. Data quality check

Before calculating metrics and running statistical tests, the dataset needs to be checked for missing values, duplicates, data types, and general structure.

This step helps confirm that the data is reliable for analysis.

In [4]:
data.info()  # Review the dataset structure, column types, and memory usage

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   user_id           10000 non-null  int64  
 1   variant           10000 non-null  object 
 2   converted         10000 non-null  int64  
 3   session_duration  10000 non-null  float64
 4   pages_visited     10000 non-null  int64  
 5   age               10000 non-null  int64  
dtypes: float64(1), int64(4), object(1)
memory usage: 468.9+ KB


In [5]:
data.duplicated().sum() # Check for duplicate rows in the dataset

np.int64(0)

In [6]:
data["user_id"].duplicated().sum()

np.int64(0)

No duplicate rows were found in the dataset.  
The user ID check confirms whether each user appears only once in the dataset.  
This is important because A/B test metrics should be calculated at the user level.

Will check the distribution between the control and treatment groups to ensure that the comparison between variants is reliable.

In [7]:
data.groupby('variant').describe()

user_id                                                         \
            count         mean          std  min     25%     50%     75%   
variant                                                                    
control    5013.0  4992.565929  2880.026441  1.0  2455.0  5003.0  7486.0   
treatment  4987.0  5008.475436  2894.051176  2.0  2538.5  4997.0  7514.0   

                   converted            ... pages_visited           age  \
               max     count      mean  ...           75%   max   count   
variant                                 ...                               
control    10000.0    5013.0  0.106523  ...           4.0  10.0  5013.0   
treatment   9999.0    4987.0  0.143373  ...           6.0  14.0  4987.0   

                                                              
               mean        std   min   25%   50%   75%   max  
variant                                                       
control    40.94215  13.446676  18.0  30.0  41.0  53.0  64.0  
treatment  40.70764  13.418603  18.0  30.0  40.0  52.0  64.0  

[2 rows x 40 columns]

#### Number of users in each group

In [8]:
group_count = data.groupby('variant')['user_id'].nunique()

In [9]:
group_percent = (group_count / group_count.sum()) * 100

In [10]:
print(group_percent)

variant
control      50.13
treatment    49.87
Name: user_id, dtype: float64


The users are almost evenly split between the two groups: 50.13% in control and 49.87% in treatment.

This indicates that the A/B test groups are well balanced.

#### Number of conversions in each group

Check how many users completed the target action.

We compare both the number of conversions and the conversion rate for each group.

In [11]:
data.groupby('variant')['converted'].sum()

variant
control      534
treatment    715
Name: converted, dtype: int64

#### Calculation of the average session duration for each variant.

Session duration is used as an engagement metric.  
It shows how long users stayed on the website during their session.

In [12]:
data.groupby('variant')['session_duration'].mean()

variant
control      5.028694
treatment    7.018729
Name: session_duration, dtype: float64

The average session duration is higher in the treatment group.  
This may indicate that users spent more time interacting with the new webpage design.

#### The average number of pages visited for each variant.

It helps evaluate whether users explored more content after seeing the new design.

In [13]:
data.groupby('variant')['pages_visited'].mean()

variant
control      2.978855
treatment    5.007419
Name: pages_visited, dtype: float64

The treatment group visited more pages on average than the control group.  
This suggests that the new design may have increased user engagement.

## 4. Statistical tests

Raw differences between groups are not enough to make a reliable conclusion.  
Statistical tests are used to determine whether the observed differences are likely to be real or could have happened by chance.

The significance level is set to 0.05.

If p-value < 0.05, the difference is considered statistically significant.  
If p-value >= 0.05, there is not enough evidence to reject the null hypothesis.

A two-proportion z-test is used to compare conversion rates between the control and treatment groups.

Null hypothesis: there is no difference in conversion rate between the two groups.  
Alternative hypothesis: there is a difference in conversion rate between the two groups.

In [14]:
control_group = data[data["variant"] == "control"]
treatment_group = data[data["variant"] == "treatment"]

control_subs = control_group["converted"].sum()
treatment_subs = treatment_group["converted"].sum()

control_users = control_group["user_id"].nunique()
treatment_users = treatment_group["user_id"].nunique()

In [15]:
from statsmodels.stats.proportion import proportions_ztest

alpha = 0.05

success = [control_subs, treatment_subs]
nobs = [control_users, treatment_users]

stat, p_value = proportions_ztest(success, nobs)

print(f"Z-stat: {stat:.4f}")
print(f"P-value: {p_value:.10f}")

if p_value < alpha:
    print("The difference is statistically significant. Null Hypothesis is rejected.")
else:
    print("The difference is not statistically significant. Null Hypothesis cannot be rejected.")

Z-stat: -5.5730
P-value: 0.0000000250
The difference is statistically significant. Null Hypothesis is rejected.


In [16]:
from scipy import stats

alpha = 0.05

def run_ttest(metric):
    statistic, pvalue = stats.ttest_ind(
        data[data["variant"] == "control"][metric],
        data[data["variant"] == "treatment"][metric],
        equal_var=False,
        nan_policy="omit",
        alternative="two-sided"
    )
    
    print(f"Metric: {metric}")
    print(f"T-statistic: {round(statistic, 4)}, p-value: {pvalue:.6f}")
    
    if pvalue < alpha:
        print("The difference is statistically significant. Null Hypothesis is rejected.")
    else:
        print("The difference is not statistically significant. Null Hypothesis cannot be rejected.")
    
    print("-" * 60)

run_ttest("session_duration")
run_ttest("pages_visited")
# Save t-test results into variables for the final results table

session_stat, session_pvalue = stats.ttest_ind(
    data[data["variant"] == "treatment"]["session_duration"],
    data[data["variant"] == "control"]["session_duration"],
    equal_var=False,
    nan_policy="omit",
    alternative="two-sided"
)

pages_stat, pages_pvalue = stats.ttest_ind(
    data[data["variant"] == "treatment"]["pages_visited"],
    data[data["variant"] == "control"]["pages_visited"],
    equal_var=False,
    nan_policy="omit",
    alternative="two-sided"
)

test_results = pd.DataFrame({
    "metric": [
        "Conversion Rate",
        "Average Session Duration",
        "Average Pages Visited"
    ],
    "test": [
        "Two-proportion z-test",
        "Welch's t-test",
        "Welch's t-test"
    ],
    "statistic": [
        stat,
        session_stat,
        pages_stat
    ],
    "p_value": [
        p_value,
        session_pvalue,
        pages_pvalue
    ]
})

test_results["significant"] = test_results["p_value"] < alpha

test_results

Metric: session_duration
T-statistic: -44.341, p-value: 0.000000
The difference is statistically significant. Null Hypothesis is rejected.
------------------------------------------------------------
Metric: pages_visited
T-statistic: -51.2558, p-value: 0.000000
The difference is statistically significant. Null Hypothesis is rejected.
------------------------------------------------------------


,metric,test,statistic,p_value,significant
0,Conversion Rate,Two-proportion z-test,-5.573048,2.503207e-08,True
1,Average Session Duration,Welch's t-test,44.340981,0.000000e+00,True
2,Average Pages Visited,Welch's t-test,51.255843,0.000000e+00,True


## 5. Age segmentation

To better understand how different user groups respond to the new design, users are divided into four age groups:

- younger than 25
- 25–34
- 35–44
- 45 and older

This helps identify whether the treatment performs consistently across different age segments.

In [17]:
bins = [0, 24, 34, 44, np.inf]
labels = ["<25", "25-34", "35-44", "45+"]

data["age_group"] = pd.cut(
    data["age"],
    bins=bins,
    labels=labels,
    right=True
)

data[["age", "age_group"]].head()

,age,age_group
0,57,45+
1,24,<25
2,49,45+
3,36,35-44
4,23,<25


In [18]:
age_summary = data.groupby(
    ["age_group", "variant"],
    observed=True
).agg(
    users=("user_id", "nunique"),
    conversions=("converted", "sum"),
    conversion_rate=("converted", "mean"),
    avg_session_duration=("session_duration", "mean"),
    avg_pages_visited=("pages_visited", "mean")
).reset_index()

age_summary

,age_group,variant,users,conversions,conversion_rate,avg_session_duration,avg_pages_visited
0,<25,control,738,73,0.098916,5.059546,2.947154
1,<25,treatment,715,105,0.146853,6.980870,4.930070
2,25-34,control,1079,122,0.113068,5.021787,2.980538
3,25-34,treatment,1141,152,0.133216,7.078861,4.844873
4,35-44,control,1075,111,0.103256,5.031442,2.956279
5,35-44,treatment,1098,170,0.154827,7.052419,5.050091
6,45+,control,2121,228,0.107496,5.020080,3.000471
7,45+,treatment,2033,288,0.141663,6.980100,5.102804


The age group summary shows performance metrics for each age segment and each test variant.

Across all age groups, the treatment group shows higher conversion rate, longer average session duration, and more pages visited compared to the control group.

## 6. Export Data for Dashboard

The prepared summary tables are exported as CSV files.  
These files will be used to build the Tableau dashboard.

In [19]:
data.to_csv("ab_test_cleaned.csv", index=False, encoding="utf-8-sig")
age_summary.to_csv("age_summary.csv", index=False, encoding="utf-8-sig")
test_results.to_csv("test_results.csv", index=False, encoding="utf-8-sig")